# NHCS Scenario 2 — The pandemic years month by month: the collapse of April 2020, the COVID-19 waves and the return of influenza

_A time axis through 1.4 million visits. ED dataset only. Level: first analysis, no statistics background needed. About 60 minutes._

**Before you start.** The NHCS pathology holds two datasets from about 200 US hospitals, 2020 to
2022: 1,368,648 emergency department visit records and 488,636 inpatient discharge records, each a
5% sample of all encounters of the participating hospitals. **A row is an encounter record, not a
patient**, and the two datasets cannot be linked: each is an independent sample, the identifiers
were renumbered, and NCHS merged the ED outcome "admitted" into "other". The MIP does not use the
survey weights: every figure describes *the sampled encounters*, not US hospital care. Codes are
NCHS codes: Sex 1 = Male, 2 = Female; NCHS flags 1 = Yes, 2 = No; curated flags 0 = No, 1 = Yes;
age is top-coded at 86 and the length of stay at 15 days.

**How this notebook works.** Every analysis is a *federated* call: the code selects a dataset and
variables, defines a cohort with filters and asks the MIP for aggregates (counts, means, a test, a
model). No record ever reaches this notebook. Run the cells in order; each step says what to look
for and what result to expect. These are the largest datasets of this MIP: the notebook prints how
long the regressions take.

**Learning goals.** Use a histogram over a month index as a time series; compute a rate per month
with one call per month; read epidemic waves and seasonality in routine data; compare years with a
chi-square; compare two groups of visits with a t-test; and say what a changing sample of hospitals
does to a trend.

**Clinical background.** The three years on the MIP are the pandemic years. In the United States,
emergency department visits fell by about half in April 2020 and recovered slowly. COVID-19
arrived in waves: spring 2020, the winter of 2020-21, the Delta wave of summer 2021, the Omicron
wave of January 2022. Influenza, which fills emergency departments every winter, almost
disappeared for two seasons under masks and distancing and came back in late 2022. NHCS records
the month of every visit and, in the diagnosis codes, COVID-19 (U07.1, a code that exists since
April 2020) and influenza. The sample of hospitals changed from year to year (191, 209 and 198),
so the level of a curve is not comparable across years as a national count would be; its shape
within a year is.

## 1. Connect to the MIP and open the NHCS pathology

In [ ]:
import mip
from mip.filters import F

client = mip.Client.from_env()          # connection configured by the MIP portal
catalog = client.catalog()

try:
    dm = catalog.data_model("NHCS")        # add version="ED_Inpatient_v1" if the catalogue holds several versions
except LookupError:
    display(catalog.summaries())           # the pathology label differs: pick it from this list
    raise
def dataset(code, label):
    # by the code, which is how the platform labels its datasets (seen 2026-10-12), else by the long label
    try:
        return dm.datasets[code]
    except (KeyError, LookupError):
        return dm.datasets[label]

ed = dataset("nhcs_ed", "NHCS emergency department visits, USA, 2020-2022")
ip = dataset("nhcs_ip", "NHCS inpatient discharges, USA, 2020-2022")
display(dm)
display(dm.datasets.to_frame())

In [ ]:
# The variable tree is the anatomy of one encounter record: Encounter, Patient, Discharge,
# Diagnoses (with the curated condition flags and the NCHS CCSR flags), and the branch
# Inpatient stay (empty for ED visits): length of stay and procedures.
dm.tree()

## 2. Toolbox

In [ ]:
# ---- Toolbox: small helpers that turn MIP results into readable tables and charts. ----
# Run this cell once. You do not need to read it to follow the scenario; come back to it
# when you want to see how a result payload is unpacked.
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from mip.preprocessing import MissingValuesHandler

POOLED = "all datasets"                 # the row describe() adds across the selected datasets
PCT = "{:.1f}"                          # number format of the percentage columns


def analysis(variables, filters=None, drop_missing=False, datasets=None):
    """A Pipeline on the datasets of this notebook (DATASETS, or the ones given) with the given
    variables and cohort filter. Tests and models need drop_missing=True: the platform requires an
    explicit missing-values step and analyses complete cases only. describe() keeps it off so it
    can count missing values."""
    handler = MissingValuesHandler(strategies={v: "drop" for v in variables}) if drop_missing else None
    return mip.Pipeline(analysis_set=dm.select(datasets=datasets or DATASETS, variables=variables),
                        filters=filters, handle_missing=handler)


def labels_of(variable):
    """{code: label} of a categorical variable, whatever format the catalogue uses."""
    raw = variable._data.get("enumerations") or {}
    if isinstance(raw, dict):
        return {str(k): str(v) for k, v in raw.items()}
    return {str(item.get("code") if isinstance(item, dict) else item):
            str(item.get("label") if isinstance(item, dict) else item) for item in raw}


def rows_for(result, variable, dataset=POOLED):
    """The describe row of one variable: the pooled row by default, or the row of one Dataset."""
    keys = (POOLED,) if dataset == POOLED else (dataset._code, dataset.label)
    return [r for r in result.raw.get("featurewise", [])
            if r.get("variable") == variable._code and r.get("dataset") in keys]


def numeric_summary(result, variables, dataset=POOLED):
    """n, missing, mean, SD, quartiles and range of numerical variables from a describe result."""
    rows = []
    for v in variables:
        for r in rows_for(result, v, dataset):
            d = r.get("data") or {}
            rows.append({"variable": v.label, "n": d.get("num_dtps"), "missing": d.get("num_na"),
                         "mean": d.get("mean"), "sd": d.get("std"), "q1": d.get("q1"), "median": d.get("q2"),
                         "q3": d.get("q3"), "min": d.get("min"), "max": d.get("max")})
    return pd.DataFrame(rows)


def category_counts(result, variable, dataset=POOLED):
    """Counts and percentages of the levels of a categorical variable from a describe result."""
    lab = labels_of(variable)
    rows = []
    for r in rows_for(result, variable, dataset):
        counts = (r.get("data") or {}).get("counts") or {}
        total = sum(counts.values())
        for c, n in counts.items():
            rows.append({"variable": variable.label, "level": lab.get(str(c), str(c)), "count": n,
                         "percent": 100 * n / total if total else np.nan, "missing": (r.get("data") or {}).get("num_na")})
    return pd.DataFrame(rows)


def level_percent(result, variable, levels=("Yes",), dataset=POOLED):
    """Percentage of the given levels (labels) of a categorical variable in a describe result."""
    k = category_counts(result, variable, dataset)
    return float(k.loc[k["level"].isin(levels), "percent"].sum()) if len(k) else np.nan


def by_level(numeric, group, levels=None, base_filter=None, datasets=None):
    """n, mean, median and quartiles of a numerical variable in each level of a categorical one
    (one describe per level). levels: the codes to use, by default every code of the variable."""
    lab = labels_of(group)
    rows = []
    for c in levels or lab:
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis([numeric], f, datasets=datasets).describe()
        for r in rows_for(res, numeric):
            d = r.get("data") or {}
            rows.append({group.label: lab.get(c, c), "n": d.get("num_dtps"), "mean": d.get("mean"),
                         "q1": d.get("q1"), "median": d.get("q2"), "q3": d.get("q3")})
    return pd.DataFrame(rows)


def band_means(numeric, bands, base_filter=None, datasets=None):
    """n, mean and median of a numerical variable in bands given as {label: filter}."""
    rows = []
    for label, f in bands.items():
        if base_filter is not None:
            f = f & base_filter
        res = analysis([numeric], f, datasets=datasets).describe()
        for r in rows_for(res, numeric):
            d = r.get("data") or {}
            rows.append({"band": label, "n": d.get("num_dtps"), "mean": d.get("mean"), "median": d.get("q2")})
    return pd.DataFrame(rows)


def crosstab(exposure, outcome, base_filter=None, event="1", levels=None, event_label=None, datasets=None):
    """Cross-tabulation of outcome by exposure, as the MIP does it: counts come from one
    describe per exposure level, the p-value from the Chi-squared Test on the whole cohort.
    `event` is the outcome code counted in the percentage column, or a list of codes."""
    lab_x, lab_y = labels_of(exposure), labels_of(outcome)
    events = [event] if isinstance(event, str) else list(event)
    pct_col = "% " + (event_label or lab_y.get(events[0], events[0]))
    rows = []
    for c in levels or lab_x:
        f = (F(exposure) == c) & F(outcome).is_not_null()
        if base_filter is not None:
            f = f & base_filter
        res = analysis([exposure, outcome], f, datasets=datasets).describe()
        counts = {}
        for r in rows_for(res, outcome):
            counts = (r.get("data") or {}).get("counts") or {}
        n = sum(counts.values())
        if n == 0:
            continue
        row = {exposure.label: lab_x.get(c, c), "n": n}
        for yc, ylabel in lab_y.items():
            row[ylabel] = counts.get(yc, 0)
        row[pct_col] = 100 * sum(counts.get(e, 0) for e in events) / n
        rows.append(row)
    table = pd.DataFrame(rows)
    f = F(exposure).is_not_null() & F(outcome).is_not_null()
    if levels:
        f = f & F(exposure).isin(list(levels))
    if base_filter is not None:
        f = f & base_filter
    chi = analysis([exposure, outcome], f, drop_missing=True, datasets=datasets).chi_square_test(x=exposure, y=outcome)
    stats = chi.raw if isinstance(chi.raw, dict) else {}
    table.attrs["chi2"] = stats.get("chi2")
    table.attrs["p"] = stats.get("p_value", stats.get("p"))
    table.attrs["outcome"] = outcome.label
    return table


def rates_by(group, outcomes, levels=None, base_filter=None, event="1", datasets=None):
    """% of records with each outcome (code `event`) in every level of `group`: one describe per
    level for the counts of all outcomes, one Chi-squared Test per outcome (in .attrs)."""
    lab = labels_of(group)
    rows = []
    for c in levels or lab:
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis(outcomes, f, datasets=datasets).describe()
        row = {group.label: lab.get(c, c)}
        for y in outcomes:
            counts = {}
            for r in rows_for(res, y):
                counts = (r.get("data") or {}).get("counts") or {}
            n = sum(counts.values())
            row.setdefault("n", n)
            row[y.label] = 100 * counts.get(event, 0) / n if n else np.nan
        rows.append(row)
    table = pd.DataFrame(rows)
    table.attrs["chi2"], table.attrs["p"] = {}, {}
    for y in outcomes:
        f = F(group).is_not_null() & F(y).is_not_null()
        if levels:
            f = f & F(group).isin(list(levels))
        if base_filter is not None:
            f = f & base_filter
        chi = analysis([group, y], f, drop_missing=True, datasets=datasets).chi_square_test(x=group, y=y)
        table.attrs["chi2"][y.label] = chi.raw.get("chi2")
        table.attrs["p"][y.label] = chi.raw.get("p_value")
    return table


def month_label(i):
    """Month index 1..36 -> '2020-01' .. '2022-12'."""
    return f"{2020 + (i - 1) // 12}-{(i - 1) % 12 + 1:02d}"


def by_month(outcomes, variable, months=range(1, 37), datasets=None):
    """% of records with each outcome (code 1) per month of the time axis: one describe per month."""
    rows = []
    for m in months:
        res = analysis(outcomes, F(variable) == m, datasets=datasets).describe()
        row = {"month": m, "label": month_label(m)}
        for y in outcomes:
            counts = {}
            for r in rows_for(res, y):
                counts = (r.get("data") or {}).get("counts") or {}
            n = sum(counts.values())
            row.setdefault("n", n)
            row[y.label] = 100 * counts.get("1", 0) / n if n else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def show_crosstab(table, title=None, rotate=0):
    """Display a crosstab table, its chi-square test and a bar chart of the percentages."""
    pct_col = [c for c in table.columns if c.startswith("% ")][0]
    display(table.style.format({pct_col: PCT}).hide(axis="index"))
    print(f"Chi-squared Test: chi2 = {table.attrs['chi2']:,.0f}, p = {fmt_p(table.attrs['p'])}")
    ax = table.set_index(table.columns[0])[pct_col].plot(kind="bar", color="#4C72B0", figsize=(7, 3))
    ax.set_ylabel(pct_col)
    ax.set_title(title or f"{table.attrs['outcome']} by {table.columns[0]}")
    ax.set_xlabel("")
    plt.xticks(rotation=rotate)
    plt.tight_layout()
    plt.show()


def anova_summary(result):
    """One-way ANOVA result -> F, p, n, a table of the group means and SDs, and the Tukey comparisons."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    table, ci = raw.get("anova_table", {}), raw.get("ci_info", {})
    groups = pd.DataFrame({"mean": pd.Series(ci.get("means", {})), "sd": pd.Series(ci.get("sample_stds", {}))})
    return table.get("f_stat"), table.get("p_value"), table.get("n_obs"), groups, pd.DataFrame(raw.get("tuckey_test", []))


def ttest_summary(result):
    """Independent t-test result -> t, p, df, mean difference with its 95% CI, Cohen's d, n."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    return {k: raw.get(k) for k in ("t_stat", "p", "df", "mean_diff", "ci_lower", "ci_upper", "cohens_d", "n_obs")}


def n_fmt(value):
    # a count with thousands separators, or n/a when the platform did not return it
    return f"{value:,}" if isinstance(value, (int, float)) else "n/a"


def ttest_from_describe(result_a, result_b, variable):
    """The pooled-variance t statistic of two groups from their describe results (n, mean, SD):
    all a t-test needs, for groups the platform's t-test cannot form (several codes pooled)."""
    a = numeric_summary(result_a, [variable]).iloc[0]
    b = numeric_summary(result_b, [variable]).iloc[0]
    n1, n2 = a["n"], b["n"]
    sp2 = ((n1 - 1) * a["sd"] ** 2 + (n2 - 1) * b["sd"] ** 2) / (n1 + n2 - 2)
    t = (a["mean"] - b["mean"]) / math.sqrt(sp2 * (1 / n1 + 1 / n2))
    return {"mean_a": a["mean"], "mean_b": b["mean"], "diff": a["mean"] - b["mean"], "t": t, "n": n1 + n2}


def term_label(term, variables):
    """Model term -> readable label: 'SEX[2]' -> 'Sex = Female', 'AGE' -> 'Age (per unit)'."""
    by_code = {v._code: v for v in variables}
    term = str(term)
    if "[" in term and term.endswith("]"):
        base, level = term[:-1].split("[", 1)
        v = by_code.get(base)
        return f"{v.label} = {labels_of(v).get(level, level)}" if v else term
    v = by_code.get(term)
    return f"{v.label} (per unit)" if v else term


def odds_ratio_table(result, variables):
    """Logistic regression result -> one row per term with odds ratio, 95% CI and p.
    Categorical covariates appear as 'variable = level' relative to the reference level
    (the first code of the variable) that the platform leaves out."""
    frame = result.to_frame().copy()
    frame = frame[~frame["feature"].astype(str).str.lower().isin(["intercept", "const"])]
    frame["term"] = frame["feature"].map(lambda t: term_label(t, variables))
    for col in ("odds_ratio", "or_lower_ci", "or_upper_ci", "p"):
        if col not in frame.columns:
            frame[col] = np.nan
    out = frame[["term", "odds_ratio", "or_lower_ci", "or_upper_ci", "p"]]
    return out.rename(columns={"or_lower_ci": "ci_low", "or_upper_ci": "ci_high"}).reset_index(drop=True)


def show_odds_ratios(table, title):
    """Table and forest plot (log scale) of odds ratios."""
    display(table.style.format({"odds_ratio": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))
    t = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.5))
    ax.errorbar(t["odds_ratio"], range(len(t)),
                xerr=[t["odds_ratio"] - t["ci_low"], t["ci_high"] - t["odds_ratio"]], fmt="o", capsize=4)
    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel("odds ratio (log scale)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def linear_table(result, variables):
    """Linear regression result -> one row per term with coefficient (units of the outcome), 95% CI and p."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    return pd.DataFrame({"term": [term_label(t, variables) for t in raw.get("indep_vars", [])],
                         "coefficient": raw.get("coefficients"), "ci_low": raw.get("lower_ci"),
                         "ci_high": raw.get("upper_ci"), "p": raw.get("pvalues")})


def show_coefficients(table, title, unit="days"):
    """Table and dot chart of linear-regression coefficients (the intercept left out of the chart)."""
    display(table.style.format({"coefficient": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))
    t = table[table["term"] != "Intercept"].iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.5))
    ax.errorbar(t["coefficient"], range(len(t)),
                xerr=[t["coefficient"] - t["ci_low"], t["ci_high"] - t["coefficient"]], fmt="o", capsize=4)
    ax.axvline(0, linestyle="--", linewidth=1)
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel(f"{unit} added, for the same values of the other predictors")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


print("Toolbox ready.")

## 3. The dataset and the variables of this scenario

| Code | Label in the MIP | Tree location (under NHCS) | Role | Values |
|---|---|---|---|---|
| `month_index` | Month index 2020-2022 | Encounter | time axis, 1 = January 2020 to 36 = December 2022; curation-derived | integer, 1-36 months |
| `YEAR` | Survey year | Encounter | grouping | 2020 2020; 2021 2021; 2022 2022 |
| `DISCHARGE_MONTH` | Discharge month | Encounter | month within the year (extension) | 12 codes, listed in the tree |
| `covid_dx` | COVID-19, any position | Diagnoses / Condition flags (curated) | COVID-19 in any position; curation-derived | 0 No; 1 Yes |
| `covid_dx1` | COVID-19 as first-listed diagnosis | Diagnoses / Condition flags (curated) | COVID-19 first-listed; curation-derived | 0 No; 1 Yes |
| `cond_influenza` | Influenza | Diagnoses / Condition flags (curated) | influenza; curation-derived | 0 No; 1 Yes |
| `cond_injury` | Injury diagnosis | Diagnoses / Condition flags (curated) | injuries by month; curation-derived | 0 No; 1 Yes |
| `age_group` | Age group | Patient | who had COVID-19; curation-derived | 1 Under 1 year; 2 1-17 years; 3 18-44 years; 4 45-64 years; 5 65-74 years; 6 75-84 years; 7 85 years and over |
| `AGE` | Age | Patient | t-test by COVID-19 | integer, 0-86 years |
| `died` | Died during encounter | Discharge | death in the ED by COVID-19; curation-derived | 0 No; 1 Yes |
| `CCSR_INF012` | CCSR INF012: Coronavirus disease 2019 (COVID-19) (ED 2021-2022) | Diagnoses / CCSR flags (NCHS) | NCHS COVID-19 flag, 2021-2022 (extension) | 1 Yes; 2 No |

Variables are picked by their **label**, as the tree shows it; the tree location tells you where
to find them in `dm.tree()` and in the MIP interface. The code (for example `AGE`) is what the csv
column is called; you will see it in some result payloads. The first line below chooses the
dataset(s) every call of this notebook uses.

In [ ]:
DATASETS = [ed]                          # every call of this notebook runs on the ED visits

month      = dm.variables["Month index 2020-2022"]
year       = dm.variables["Survey year"]
dmonth     = dm.variables["Discharge month"]
covid      = dm.variables["COVID-19, any position"]
covid1     = dm.variables["COVID-19 as first-listed diagnosis"]
influenza  = dm.variables["Influenza"]
injury     = dm.variables["Injury diagnosis"]
age_group  = dm.variables["Age group"]
age        = dm.variables["Age"]
died       = dm.variables["Died during encounter"]
[ccsr_covid] = dm.variables.search("CCSR INF012")
display(mip.to_frame([month, year, dmonth, covid, covid1, influenza, injury, age_group, age, died, ccsr_covid]))

## Step 1. Visits per month: the collapse of April 2020

A histogram of the month index with 36 bins gives the number of visits of every month in one
federated call.

*Expected:* 44,757 visits in January 2020, 40,784 in February, 35,874 in March, then 20,942 in
April 2020, less than half of January; 25,203 in May, 27,892 in June, and 30,000 to 34,000 for the
rest of 2020. In 2021 the months run from 31,836 (February) to 44,420 (August); in 2022 from
33,567 (February) to 44,702 (November). Every February is low, every summer high. In April 2020
half of the usual visits did not happen; the curve never fully returned to its January 2020 level
in this sample, but part of that is the changing set of hospitals.

In [ ]:
h = analysis([month], drop_missing=True).histogram(variable=month, bins=36).raw["histogram"][0]
volume = pd.Series([c or 0 for c in h["counts"]], index=[month_label(i) for i in range(1, 37)], name="visits")
display(volume.to_frame().T)
print(f"January 2020: {volume.iloc[0]:,}; April 2020: {volume.iloc[3]:,} ({100 * volume.iloc[3] / volume.iloc[0]:.0f}% of January); "
      f"lowest month {volume.idxmin()} ({volume.min():,}), highest {volume.idxmax()} ({volume.max():,})")
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.bar(volume.index, volume.values, color=["#C44E52" if i < 12 else "#4C72B0" for i in range(36)])
ax.set_ylabel("visits"); ax.set_title("Sampled ED visits per month, 2020 (red) to 2022")
ax.set_xticks(range(0, 36, 3)); ax.set_xticklabels(volume.index[::3], rotation=45, fontsize=8)
plt.tight_layout()
plt.show()

## Steps 2 and 3. COVID-19, influenza and injuries by month

One `describe()` per month of four flags: 36 federated calls, each on about 35,000 visits.

*Expected, COVID-19 in any position:* 0.0% in January to March 2020 (the code did not exist),
6.4% in April 2020, then 1 to 3% through the summer, 6.0% in November and 8.8% in December 2020,
7.6% in January 2021; 0.8% in June 2021; 6.4% in August 2021 (Delta); 10.9% in December 2021 and
17.8% in January 2022 (Omicron); 1.2% in March 2022; 3 to 6% for the rest of 2022. First-listed
COVID-19 follows the same waves at about two thirds of the level.

*Expected, influenza:* 4.5% in January and 4.6% in February 2020, 2.2% in March, 0.3% in April,
then 0.0 to 0.1% every month until September 2022 (one short bump to 1.4% in April 2022); 1.6%
in October, 5.9% in November and 5.4% in December 2022.

*Expected, injuries:* 13 to 15% of visits in the winter months, 17 to 19% from May to September
of each year; 14.9% in April 2020.

In [ ]:
t0 = time.perf_counter()
monthly = by_month([covid, covid1, influenza, injury], month).set_index("label")
print(f"36 federated calls in {time.perf_counter() - t0:.0f} s")
display(monthly.drop(columns="month").style.format({c: PCT for c in monthly.columns if c not in ("month", "n")}))
peaks = monthly[covid.label]
print("COVID-19 peaks: " + ", ".join(f"{m} {v:.1f}%" for m, v in peaks[peaks > 6].items()))

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
monthly[[covid.label, covid1.label, influenza.label]].plot(ax=axes[0], marker="o", color=["#C44E52", "#DD8452", "#4C72B0"])
axes[0].set_ylabel("% of visits"); axes[0].set_title("COVID-19 and influenza in the sampled ED visits, by month"); axes[0].legend(frameon=False, fontsize=8)
monthly[injury.label].plot(ax=axes[1], marker="o", color="#55A868")
axes[1].set_ylabel("% of visits"); axes[1].set_title("Injury diagnoses by month"); axes[1].set_ylim(0)
axes[1].set_xticks(range(0, 36, 3)); axes[1].set_xticklabels(monthly.index[::3], rotation=45, fontsize=8); axes[1].set_xlabel("")
plt.tight_layout()
plt.show()

*What it means:* routine hospital data record the epidemic as it happened, wave by wave, with the
dates known from the news. The disappearance of influenza for two winters is one of the clearest
effects of distancing in any dataset; its return in late 2022 was early and strong. Injuries
follow activity, with a summer peak every year.

## Step 4. COVID-19 by year

The cross-tabulation of the two COVID-19 flags by survey year.

*Expected:* COVID-19 in any position in 2.6% of the visits of 2020, 4.6% of 2021, 4.7% of 2022
(chi-square 2,981, p < 0.001); as first-listed diagnosis 1.8%, 3.3% and 3.2%. 2020 is lower partly
because the code only existed from April, and partly because testing was scarce in the first
months. A yearly rate hides the waves.

In [ ]:
by_year = rates_by(year, [covid, covid1]).set_index("Survey year")
display(by_year.style.format({c: PCT for c in by_year.columns if c != "n"}))
for label, chi2 in by_year.attrs["chi2"].items():
    print(f"{label}: chi2 = {chi2:,.0f}, p = {fmt_p(by_year.attrs['p'][label])}")

## Step 5. Who came with COVID-19

COVID-19 by age group, death in the ED by COVID-19, and the *Independent t-test* of age by
COVID-19.

*Expected:* COVID-19 was coded in 6.2% of the visits of infants under 1 year, 2.9% at 1 to 17,
3.7% at 18 to 44, 4.6% at 45 to 64 and 4.8 to 4.9% at 65 and over (chi-square 1,900, p < 0.001).
Visits with COVID-19 were older on average, 45.1 against 41.5 years (t about 34, p < 0.001), and
ended in death in the ED in 2.3% against 0.4% of the other visits (1,224 of 53,868 COVID-19
visits; chi-square 3,313, p < 0.001). The infant figure surprises: fever in an infant brings the
family to the ED and leads to testing. Death in the ED is rare in any case; the mortality of
COVID-19 belongs to the inpatient dataset (Scenario 4).

In [ ]:
show_crosstab(crosstab(age_group, covid, event_label="with COVID-19"), "COVID-19 among the diagnoses, by age group", rotate=30)
show_crosstab(crosstab(covid, died, event_label="died in the ED"), "Death in the ED, by COVID-19 diagnosis")
t = analysis([age, covid], drop_missing=True).t_test(variable=age, group_by=covid, group_a="1", group_b="0")
s = ttest_summary(t)
print(f"Age, COVID-19 minus other visits: {s['mean_diff']:.1f} years (95% CI {s['ci_lower']:.1f} to {s['ci_upper']:.1f}), "
      f"t = {s['t_stat']:.1f}, p = {fmt_p(s['p'])}, n = {n_fmt(s['n_obs'])}")

## Step 6. State the result

A model answer:

> In the sampled visits of about 200 US emergency departments, the monthly volume fell from 44,757
> in January 2020 to 20,942 in April 2020 and recovered through 2021. COVID-19 appears in the
> diagnoses from April 2020 and peaks in December 2020 (8.8% of visits), August 2021 (6.4%) and
> January 2022 (17.8%); influenza, 4.5% of visits in January 2020, vanishes from April 2020 to
> September 2022 and returns to 5.9% in November 2022. Injuries rise every summer. Visits with
> COVID-19 were older (45 against 42 years) and died in the ED five times more often (2.3% against
> 0.4%). These are unweighted sample statistics from a changing set of hospitals.

**Questions for discussion.** The monthly curve of 2021 and 2022 stays below January 2020: give
two explanations, one about patients and one about the survey. Why does the first-listed COVID-19
rate run at two thirds of the any-position rate? What else would you expect to have disappeared
from the ED while influenza did, and what to have increased? Does the infants' COVID-19 rate, the
highest of all age groups, mean infants were the most infected?

**Optional extension.** The NCHS COVID-19 flag, delivered for 2021 and 2022 only, against the
curated flag: both give about 4.6 to 4.8% in those years.

In [ ]:
nchs = crosstab(year, ccsr_covid, levels=["2021", "2022"], event_label="with the NCHS COVID-19 flag")
display(nchs[[year.label, "n", nchs.columns[-1]]].style.format({nchs.columns[-1]: PCT}).hide(axis="index"))